In [1]:
import os
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from utils.logistic_lasso_regression_fista import LogisticLassoRegressionFISTA
from utils.missing_schemas import MCAR, MNAR, MAR1, MAR2
from sklearn.neighbors import KNeighborsClassifier

DATA_DIR = "../data/.ipynb_checkpoints/"
SPAM_DIR = "spambase"

### Generating a dictionary with different missing labels schema and test data

In [5]:
def generate_missing_label_data(X,y, missing_rate):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,train_size=0.8)
    scaler = MinMaxScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    #applying missing label schemas to train data
    X_mcar, y_mcar = MCAR(X_train, y_train, missing_rate)
    X_mar1, y_mar1 = MAR1(X_train, y_train, missing_rate)
    X_mar2, y_mar2 = MAR2(X_train, y_train, missing_rate)
    X_mnar, y_mnar = MNAR(X_train, y_train, missing_rate)
    missing_label_data = {   
        "MCAR": {
            "X_train": X_mcar,
            "y_train" : y_mcar
        },
        "MAR1": {
            "X_train": X_mar1,
            "y_train" : y_mar1
        },
        "MAR2": {
            "X_train": X_mar2,
            "y_train" : y_mar2
        },
        "MNAR": {
            "X_train": X_mnar,
            "y_train" : y_mnar
        },
        "test": {
            "X_test" : X_test,
            "y_test" : y_test
        }
    }
    return missing_label_data

### Approach 1 to dealing with missing labels - predicting missing labels using kNN based on the present data

In [23]:
def unlabeled_log_reg_approach_1(X_train, y_train, X_test, y_test, n_neighbors, label_type = 'proba'):
    #-1 => missing label
    X_missing = X_train[y_train == -1]
    
    y_not_missing = y_train[y_train != -1]
    X_not_missing = X_train[y_train != -1]
    
    #fitting a knn classifier to the not missing data to predict labels for the missing label instances
    knn = KNeighborsClassifier(n_neighbors = n_neighbors, weights = 'distance')
    knn.fit(X_not_missing, y_not_missing)
    
    #predicting the labels for missing label instances
    if label_type == 'hard':
        labels = knn.predict(X_missing)
    elif label_type == 'proba':
        labels = knn.predict_proba(X_missing)[:, 1]
    else:
        raise ValueError("label_type needs to be equal to 'proba' or 'hard'")
    
    #connecting the two groups to train the logistic regression model
    X_combined = np.vstack([X_not_missing, X_missing])  # stack rows vertically
    y_combined = np.concatenate([y_not_missing, labels])
    
    #creating and fitting the logistic regression model
    fista = LogisticLassoRegressionFISTA(
        lambda_=0.01,
        max_iter=1000,
        stop_condition=1e-6)
    fista.fit(X_train=X_combined, y_train=y_combined)
    
    #final prediction
    y_proba = fista.predict_proba(X_test = X_test)
    y_pred = (y_proba >= 0.5).astype(int)
    
    print(classification_report(y_test, y_pred))
    print(f"accuracy: {accuracy_score(y_test, y_pred)}")

#### Testing the kNN aproach on the spam dataset

In [24]:
X_spam = pd.read_csv(os.path.join(DATA_DIR, SPAM_DIR, "X.csv"))
y_spam = pd.read_csv(os.path.join(DATA_DIR, SPAM_DIR, "Y.csv")).iloc[:, 0]

missing_label_data = generate_missing_label_data(X_spam, y_spam, 0.1)
unlabeled_log_reg_approach_1(missing_label_data["MCAR"]["X_train"], missing_label_data["MCAR"]["y_train"], 
                             missing_label_data["test"]["X_test"], missing_label_data["test"]["y_test"],
                            n_neighbors = 15, label_type = 'proba')

              precision    recall  f1-score   support

           0       0.67      0.93      0.78       564
           1       0.72      0.28      0.40       357

    accuracy                           0.68       921
   macro avg       0.69      0.60      0.59       921
weighted avg       0.69      0.68      0.63       921

accuracy: 0.6775244299674267
